In [85]:
import glob, sys, os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

import xarray as xr
import datetime as dt

thismodule = sys.modules[__name__]


In [18]:
cases_list_file = '/home/bfildier/analyses/FildierSaba2026/input/cases.csv'
cases_df = pd.read_csv(cases_list_file,sep=';')

In [4]:
cases_list

,Old ID,New ID,Description,Satellite,Start,End,LonMin,LonMax,LatMin,LatMax
0,C1,R1,Wave packets,GOES-W,2016-06-12T18:00,2016-06-19T04:00,-180,-130,0,30
1,C2,R2,Wave packets,GOES-W,2016-06-17T09:00,2016-06-22T12:00,-180,-130,0,20
2,C3,R3,Wave packets,GOES-W,2016-06-20T00:00,2016-06-28T00:00,-180,-130,0,20
3,C4,R4,Arrays of wave packets,GOES-W,2016-08-10T00:00,2016-08-18T13:00,-180,-110,0,20
4,C5,R5,Arrays of wave packets,GOES-W,2016-09-02T00:00,2016-09-07T09:00,-180,-120,0,20
5,C6,R6,Large aggregate,GOES-W,2016-09-08T14:00,2016-09-13T19:00,-180,-140,0,30
6,C7,R7,Multiple growths,GOES-W,2016-06-05T12:00,2016-06-09T12:00,-140,-115,2,18
7,C8,R8,Large westward‑propagating aggregate,GOES-W,2016-06-26T20:00,2016-07-02T20:00,-135,-105,2,18
8,C9,R9,Upscale growth into large MCS,MSG,2016-06-03T10:00,2016-06-05T10:00,5,40,-5,15
9,C10,R10,Upscale growth into large MCS,MSG,2016-07-01T09:00,2016-07-03T09:00,5,40,-5,15


In [90]:
data_path = '/homedata/osaba/clustering/satellite_data'
data_tb = xr.open_dataarray(os.path.join(data_path,'bt_mosaic.nc'))
data_selected = xr.open_dataarray(os.path.join(data_path,'QC_interp_selected.nc'))
data_final_labels = xr.open_dataarray(os.path.join(data_path,'CCS_final_labels.nc'))

In [6]:
! ls /homedata/osaba/clustering/satellite_data

bt_mosaic_g1.nc  CCS_final_labels.nc	CC_stats_indexed.nc
bt_mosaic.nc	 CCS_initial_labels.nc	QC_interp_selected.nc


In [91]:
data_final_labels

<xarray.DataArray 'labels_3d_algo_reconstitution' (time: 1968, latitude: 1501, longitude: 9000)>
[26585712000 values with dtype=int32]
Coordinates:
  * time       (time) datetime64[ns] 2016-08-01 ... 2016-09-10T23:30:00
  * latitude   (latitude) float32 -30.0 -29.96 -29.92 ... 29.92 29.96 30.0
  * longitude  (longitude) float32 -180.0 -180.0 -179.9 ... 179.8 179.9 179.9

In [24]:
data_tb

<xarray.Dataset>
Dimensions:                 (latitude: 1501, longitude: 9000, time: 1968)
Coordinates:
  * time                    (time) datetime64[ns] 2016-08-01 ... 2016-09-10T2...
  * latitude                (latitude) float32 -30.0 -29.96 ... 29.96 30.0
  * longitude               (longitude) float32 -180.0 -180.0 ... 179.9 179.9
Data variables:
    Harmonized_irBT_mosaic  (time, latitude, longitude) float32 ...
Attributes:
    ddeg:                    0.04
    final_phase_minute:      0
    freq:                    30min
    lat_range:               (-30.0, 30.0)
    lon_range:               (-180.0, 180.0)
    mosaic_cache_hash:       231d7b28470c7a0579bf93e226be3ab7
    mosaic_penalty_interp:   10
    mosaic_penalty_nan:      1000
    mosaic_penalty_outside:  500
    mosaic_sat_order:        GOES_E,GOES_W,HIMAWARI,IODC,MSG
    source_hash:             231d7b28470c7a0579bf93e226be3ab7

In [ ]:
case = 'C19'
i_t = 30

# extract data associated to specific case
case_line = cases_df[cases_df['Old ID'] == case]

# select data between start and end 
# data_tb_case = data_tb.loc[dict(time=slice(case_line.Start.iloc[0],case_line.End.iloc[0]))]
data_tb_case = data_tb.loc[case_line.Start.iloc[0]:case_line.End.iloc[0],\
                           case_line.LatMin.iloc[0]:case_line.LatMax.iloc[0],\
                            case_line.LonMin.iloc[0]:case_line.LonMax.iloc[0]]

In [96]:
data_tb_case.time.size


193

In [ ]:
fig,axs = plt.subplots(nrows=5,ncols=5,figsize=(17.5,17.5))

for i_c in range(25):

    if i_c == 19:

        case = 'C%d'%i_c
        i_t = 120

        # extract data associated to specific case
        case_line = cases_df[cases_df['Old ID'] == case]

        # select data between start and end 
        # data_tb_case = data_tb.loc[dict(time=slice(case_line.Start.iloc[0],case_line.End.iloc[0]))]
        data_tb_case = data_tb.loc[case_line.Start.iloc[0]:case_line.End.iloc[0],\
                                case_line.LatMin.iloc[0]:case_line.LatMax.iloc[0],\
                                    case_line.LonMin.iloc[0]:case_line.LonMax.iloc[0]]

        ax = axs.flatten()[i_c]

        ax.imshow(data_tb_case[i_t])